In [30]:
from superfv import (
    BC,
    HydroSolver,
    MUSCL_SlopeLimiter,
    TimeIntegrator,
    ics,
)
import spd.initial_conditions as ic
from spd.sdfb_simulator import SPD_Simulator

from functools import partial
import numpy as np

gamma = 5/3
P0 = 1
target_time = 1.95

In [19]:
NDOF = 192
p = 3

Nelements = NDOF // (p + 1)

sim1 = SPD_Simulator(
    p=p,
    N=(Nelements // 4, Nelements),
    xlim=(0.0, 0.25),
    ylim=(0.0, 1.0),
    gamma=gamma,
    BC=(
        ("periodic", "periodic"),
        ("reflective", "reflective"),
    ),
    init_fct=ic.RTI(P0=P0, gamma=gamma),
    cfl_coeff={3: 0.4, 7: 0.2}[p],
    use_cupy=True,
    time_integrator="rk3",
    scheme="SDFB",
    fallback="MUSCL",
    slope_limiter="moncen",
    potential=True,
    limiting_variables=[0, 1, 2, 4],
    PAD=True,
    SED=True,
    blending=False,
)

In [20]:
sim1.perform_time_evolution(target_time)

t=1.95, steps taken 3908, time taken 144.27, bzcps = 0.0


In [25]:
NDOF = 192
p = 3

Nelements = NDOF // (p + 1)

sim2 = SPD_Simulator(
    p=p,
    N=(Nelements // 4, Nelements),
    xlim=(0.0, 0.25),
    ylim=(0.0, 1.0),
    gamma=gamma,
    BC=(
        ("periodic", "periodic"),
        ("reflective", "reflective"),
    ),
    init_fct=ic.RTI(P0=P0, gamma=gamma),
    viscosity=True,
    nu=1e-14,
    cfl_coeff={3: 0.4, 7: 0.2}[p],
    use_cupy=True,
    time_integrator="rk3",
    scheme="SDFB",
    fallback="MUSCL",
    slope_limiter="moncen",
    potential=True,
    limiting_variables=[0, 1, 2, 4],
    PAD=True,
    SED=True,
    blending=False,
)

In [26]:
sim2.perform_time_evolution(target_time)

t=1.95, steps taken 3908, time taken 202.433, bzcps = 0.0


In [27]:
np.max(np.abs(sim2.dm.W_cv - sim1.dm.W_cv))

np.float64(4.364103967091637e-09)

In [34]:
def gravity(idx, u, *, xp):
    gx = 0.0
    gy = 1.0

    out = xp.zeros_like(u)
    out[idx("mx")] = u[idx("rho")] * gx
    out[idx("my")] = u[idx("rho")] * gy
    out[idx("E")] = u[idx("mx")] * gx + u[idx("my")] * gy
    return out

sim1 = HydroSolver(
    ic=partial(ics.rayleigh_taylor, gamma=gamma, P0=P0),
    gamma=gamma,
    rho_min=1e-10,
    P_min=1e-10,
    source=gravity,
    nx=NDOF // 4,
    ny=NDOF,
    xlims=(0.0, 0.25),
    ylims=(0.0, 1.0),
    p=p,
    bcy=(BC.REFLECTIVE, BC.REFLECTIVE),
    use_MOOD=True,
    use_NAD=True,
    use_SED=True,
    blend_troubles=False,
    MUSCL_limiter=MUSCL_SlopeLimiter.MONCEN,
    cupy=True,
)
sim1.run(target_time, time_integrator=TimeIntegrator.SSPRK3)

SuperFV: 1947 steps | t=1.95e+00/1.95e+00, dt=6.71e-04 | rho_min=8.66e-01 | E_cons=1.86e+02 | wall=2.94e+01s (done)


In [43]:
sim2 = HydroSolver(
    ic=partial(ics.rayleigh_taylor, gamma=gamma, P0=P0),
    gamma=gamma,
    nu=1e-14,
    rho_min=1e-10,
    P_min=1e-10,
    source=gravity,
    nx=NDOF // 4,
    ny=NDOF,
    xlims=(0.0, 0.25),
    ylims=(0.0, 1.0),
    p=p,
    bcy=(BC.REFLECTIVE, BC.REFLECTIVE),
    use_MOOD=True,
    use_NAD=True,
    use_SED=True,
    blend_troubles=False,
    MUSCL_limiter=MUSCL_SlopeLimiter.MONCEN,
    cupy=True,
)
sim2.run(target_time, time_integrator=TimeIntegrator.SSPRK3)

SuperFV: 1947 steps | t=1.95e+00/1.95e+00, dt=7.11e-04 | rho_min=8.66e-01 | E_cons=1.86e+02 | wall=4.23e+01s (done)


In [44]:
np.abs(np.max(sim2.snapshot_history[-1].u - sim1.snapshot_history[-1].u))

np.float64(0.0)